In [1]:
  # -*- coding: utf-8 -*-
"""
Created on Wed Jun 24 11:36:50 2020

@author:
    - Santiago González Gallego
    - Lina Steffania

    Código en proceso (código actual)
"""

#%% Packages and global changes

from IPython import get_ipython
get_ipython().magic('reset -sf')

import os

clear = lambda: os.system('cls')  # On Windows System
clear()

import numpy as np
import math as mt
import pandas as pd
import matplotlib.pyplot as plt
import pdb
from scipy.optimize import fsolve
import time
import numba
from numba.typed import List
import numba.types as nt
njit = numba.njit

start = time.process_time()

plt.rcParams["font.family"]="Times New Roman"


#%% Inicio de solución

T = 8000     # Número de tandas
h_eu = 0.1  # Time differential for euler solution

#%% Data storage

TotalTime = int(T*h_eu)  # total time of simulation

#%%



#%%

@njit
def Spray(T,MovVarTemp_1,TEpTemp_1,Yp_1):

    #T_Eout = np.zeros(TotalTime)

    T_ECT  = []
    T_Eout = []
    Y_Eout = []
    Z_Mout = []
    P_inMV = []
    m_LMV  = []
    R_time = []

    Mov_VarTemp = Mov_VarTemp_1
    T_EpTemp    = T_EpTemp_1
    Y_p         = Y_p_1
    M_wDL       = M_wDL_1

    f_param = 1


    T_outh = np.zeros(T)
    P_h = np.zeros(T)
    time = np.zeros(T)
    m_outh = np.zeros(T)
    T_wall = np.zeros(T)
    m_ngas = np.zeros(T)
    T_duct = np.zeros(T)
    m_duct = np.zeros(T)
    Delay_j = np.zeros(2)

    T_outh[0] = 25
    P_h[0] = 101325
    T_wall[0] = 200


    for j in range(0,T):



        #%% Experimental data of the droplet distribution size.
        # This data is showed just to compare the calculated data.


        q_n = 2.09     # This parameter was obtained experimentally.
        D_n = 70.5e-6  # This parameter was obtained experimentally.

        GammaF = mt.gamma(1 - 1/q_n)

        SMD_exp = D_n*(GammaF**(-1))
        MMD_exp = D_n*((0.693)**(1/q_n))

        MMD_SMD = (0.693**(1/q_n))*GammaF

        # Reference: Modelling quality in spray drying - Kieviet.


        #%% Constants and initial conditions


        ## Design parameters:

        d0  = 0.711e-3     # [m] orifice diameter
        Ds  = 2.4*d0       # [m] swirl chamber diameter
        A_p = 0.50*d0*Ds   # [m^2] total inlet ports area

        l0 = 0.1*d0        # [m] orifice length
        Ls = 0.5*Ds        # [m] length of parallel portion of swirl chamber

        r_0 = d0/2         # [m] radius of final discharge orifice
        R_s = Ds/2         # [m] radius of swirl chamber

        # Reference (d0): Modelling quality in spray drying - Kieviet.
        # Reference: some parameters are approximated by the mean of different nozzles
        # These values comes from "Atomization and sprays - Lefebvre"


        ## Liquid parameters:

        # Control disturbance

        if j <= 200000:
            x_R = 0.46           # [-] solute fraction
        elif 200000 < j <= 325000 :
            x_R = 0.46
        else:
            x_R = 0.46

        T_R0 = 52.5            # [] refined temperature
        Tens = 56*10**(-3)     # [N/m] surface tension
        mu_L = 1.8*10**(-3)    # [kg/(m s)] viscosity of milk

        #### If the inlet liquid pressure is changed, this line must stay
        #### in the code:

        P_in  = 9*10**5       # [Pa] liquid pressure (inlet) 2.4 - 6.3 (8.5)



        P_out = 101325.        # [Pa] liquid pressure (outlet)
        P_L   = P_in - P_out   # [Pa] pressure difference

        rho_M0  = (1.635 - 0.0026*T_R0 + 2*10**(-5)*T_R0**2)
        rho_W0  = (1.0020825 - 1.14*10**(-4)*T_R0 - 3.325*10**(-6)*T_R0**2)*1000
        rho_p0 = 100*1000/(100*(1-x_R)*(1/rho_M0 - 1/(rho_W0/1000)) + 100/(rho_W0/1000))

        # Reference (x_R, T_R0, pressures): "Modelling quality in spray drying - Kieviet."
        # Reference (densities): "Modelling and control of a spray drying process - Shayantharan."


        # Air parameters:

        # T_EC  = 80                          # [°C] Extract temperature

        ### If the flow is changed as the MV, the temperature must be
        ### mantained constant and the temperature controller commented:
        #T_EC = 170
        #T_E0 = T_EC

        ######## Validation 1

        #%% heating system and duct

        T_Ref = 0

        T_amb = 25  # Ambiental Temperature

        Cp_ah   = (969.542 + 6.801*(T_amb+273.15)*1e-2 + 16.569*((T_amb+273.15)**2)*1e-5 - 67.828*((T_amb+273.15)**3)*1e-9)/1000    # [kJ/(kg K)] La T es en K. Art 6

        C_EntAh1 = (969.542/1000)*(T_amb - T_Ref)
        C_EntAh2 = ((6.801*1e-2)/1000)*( ((T_amb+273.15)**2)/2 - ((T_Ref+273.15)**2)/2 )
        C_EntAh3 = ((16.569*1e-5)/1000)*( ((T_amb+273.15)**3)/3 - ((T_Ref+273.15)**3)/3 )
        C_EntAh4 = ((-67.828*1e-9)/1000)*( ((T_amb+273.15)**4)/4 - ((T_Ref+273.15)**4)/4 )

        H_airinh = C_EntAh1 + C_EntAh2 + C_EntAh3 + C_EntAh4 ;

        R_h = 8.314/29

        m_inh = 1710  # [kg/h]
        V_h = 4  # [m^3]

        if j < 20000:
            m_ngas[j] = 6
        else:
            if j%6000 == 0:
                if j < 500000:
                    SP_h = 170
                elif 500000 <= j< 2500000:
                    SP_h = 160
                elif 2500000 <= j < 4500000:
                    SP_h = 180
                elif 4500000 <= j < 6500000:
                    SP_h = 170
                else:
                    SP_h = 170

                K_0h = (170-25)/(6-0)
                tao_h = 40
                t_0h = 4
                K_ch = 0.01*tao_h/(t_0h*K_0h)

                m_ngas[j] = m_ngas[j-1] + K_ch*(SP_h - T_outh[j])

            else:
                m_ngas[j] = m_ngas[j-1]

        C_EntAho1 = (969.542/1000)*(T_outh[j] - T_Ref)
        C_EntAho2 = ((6.801*1e-2)/1000)*( ((T_outh[j]+273.15)**2)/2 - ((T_Ref+273.15)**2)/2 )
        C_EntAho3 = ((16.569*1e-5)/1000)*( ((T_outh[j]+273.15)**3)/3 - ((T_Ref+273.15)**3)/3 )
        C_EntAho4 = ((-67.828*1e-9)/1000)*( ((T_outh[j]+273.15)**4)/4 - ((T_Ref+273.15)**4)/4 )

        H_airout = C_EntAho1 + C_EntAho2 + C_EntAho3 + C_EntAho4

        #%Q = UA*(T_wall(j)-T_out(j)) ;

        h_comb = 48.5*1000  # ; % [kJ/kg]
        Q = 0.85*m_ngas[j]*h_comb

        Cvair=Cp_ah-R_h

        P_duct = 101325  #; % [Pa]
        P_oper_r = 1.5

        C = m_inh/(P_duct*((P_oper_r*(P_oper_r-1))**0.5))

        m_outh[j] = C*((P_h[j]*(P_h[j]-P_duct))**0.5)

        Massb_1 = ((T_outh[j]+273.15)*R_h*1000/V_h)*(m_inh-m_outh[j])

        dCvdT = (6.801*1e-2 + 2*16.569*(T_amb+273.15)*1e-5 - 3*67.828*((T_amb+273.15)**2)*1e-9)/1000

        Energb_1 = (Q + m_inh*H_airinh - m_outh[j]*H_airout)/(dCvdT*P_h[j]*V_h/(1000*R_h))

        dPdt = (Massb_1 + P_h[j]*Energb_1/(T_outh[j]+273.15))/(1 + Cvair/(dCvdT*(T_outh[j]+273.15)))

        dTdt = Energb_1 - (Cvair*dPdt/(P_h[j]*dCvdT))

        P_h[j+1] = P_h[j]+h_eu*dPdt/3600
        T_outh[j+1]=T_outh[j]+h_eu*dTdt/3600
        time[j+1] = time[j] + h_eu

        ### Ducto

        Dist_duct = 9
        A_duct = 0.45*0.45
        #rho_Airh = 1.293*(273.15)/(273.15+T_outh[j])
        #Vel_duct = m_out(j)/(3600*rho_Airh*A_duct) ; % [m/s]

        if j <= 2:
            Delay = 0
            Delayp = 0
        else:
            Vel_duct = m_outh[j]/(3600*1.293*((273.15)/(273.15+T_outh[j]))*A_duct)  #; % [m/s]
            Delay = (Dist_duct)/(h_eu*Vel_duct)

            Vel_ductp = m_outh[j-1]/(3600*1.293*((273.15)/(273.15+T_outh[j-1]))*A_duct)
            Delayp = (Dist_duct)/(h_eu*Vel_ductp)


        Delay_j[0] = mt.ceil(Delayp)
        Delay_j[1] = mt.ceil(Delay)

        Pos_j1 = j - 1 + Delay_j[0]
        Pos_j2 = j + Delay_j[1]

        if Pos_j2 < T :

            if j == 1:
                Pos_j1=0

            for h in range(int(Pos_j1),int(Pos_j2)):
                T_duct[h] = T_outh[j-1]

            T_duct[int(Pos_j2)] = T_outh[j]


        if j < 100000:
            T_EC = 170.3
        else:
            T_EC = T_duct[j]
        T_E0 = T_EC

        ######## Inlet temperature controller:

        # if j <= 80000:
        #     T_EC = 160
        #     T_E0 = T_EC
        # else:
        #     T_EC = 180
        #     T_E0 = T_EC

        # else:

        #     if j <= 80000:
        #         T_EC = 160
        #         T_E0 = T_EC

        #     else:
        #     # Propuesta de control:

        #         if 85000 < j <= 200000:

        #             SP = 85

        #         else:

        #             SP = 85

        #         T_EC_ss = T_ECT[-1]   # = T_EC

        #         K_c = 0.1

        #         T_EC = T_EC_ss + K_c*(SP - T_Eout[-1])

        #         if T_EC <= 80:
        #             T_EC = 80

        #         if T_EC >= 220:
        #             T_EC = 220

        #         T_E0 = T_EC


        #### end of the temperature controller

        #### Inference controller - T_in MV - Z_out CV:

        # if j <= 80000:
        #     T_EC = 160
        #     T_E0 = T_EC

        # else:

        #     if j <= 85000:
        #         T_EC = 160
        #         T_E0 = T_EC

        #     else:
        #     # Propuesta de control:

        #         if 85000 < j <= 200000:

        #             #SP = 85
        #             SP = 0.300

        #         else:

        #             #SP = 85
        #             SP = 0.300

        #         T_EC_ss = T_ECT[-1]   # = T_EC

        #         K_0 = (0.19 - 0.31)/(180-160)
        #         tao = 11
        #         t_0 = 1

        #         K_c = 0.01*tao/(t_0*K_0)

        #         T_EC = T_EC_ss + K_c*(SP - Z_Mout[-1])

        #         if T_EC <= 80:
        #             T_EC = 80

        #         if T_EC >= 220:
        #             T_EC = 220

        #         T_E0 = T_EC


        #### end of the temperature controller


        rho_A = 1.293*(273.15)/(273.15+T_EC)   # [kg/m^3] aire density

        # Reference (T_EC): "Modelling quality in spray drying - Kieviet."
        # Reference (rho_A): "."


        #%% Velocity


        K   = A_p/(Ds*d0)
        K_v = (0.00367*K**0.29)*(P_L*rho_p0/mu_L)**0.2
        Vel = K_v*((2*P_L/rho_p0)**0.5)

        # Reference: "Atomization and sprays - Lefebvre."


        #%% Discharge coefficient and flow


        C1 = d0*rho_p0*Vel/mu_L
        C2 = l0/d0
        C3 = Ls/Ds
        C4 = A_p/(Ds*d0)
        C5 = Ds/d0

        C_D = 0.45*(C1**(-0.02))*(C2**(-0.03))*(C3**0.05)*(C4**0.52)*(C5**0.23)

        A_0 = mt.pi*(d0/2)**2  # [m^2]

        Param_m_L = 0.802

        m_L = Param_m_L*C_D*A_0*rho_p0*Vel/K_v     # [kg/s]

        m_Lconv = m_L*3600*1000/rho_p0   # [L/h]

        # The m_L is related to the Bernoulli equation.
        # Reference: "Atomization and sprays - Lefebvre."


        #%% Thick film and X


        FNum = m_L/((rho_p0**0.5)*(P_L**0.5))

        t_f = 3.66*(d0*FNum*mu_L/((P_L*rho_p0)**0.5))**0.25

        X = ((d0 - 2*t_f)**2)/(d0**2)

        # The FNum comes from its definition. Some correlations for this value
        # exists, but are really close to its definition equation.
        # Reference: "Atomization and sprays - Lefebvre."


        #%% Cone angle


        theta_2 = (16.156/2)*(K**(-0.39))*(d0**(1.13))*(mu_L**(-0.9))*(P_L**0.39)
        theta = theta_2 - 360

        if theta*180/mt.pi > 90:
            theta_g = 80
            theta = theta_g*mt.pi/180

        #theta_g = theta*180/mt.pi

        # Reference: "Atomization and sprays - Lefebvre."


        #%% Velocity components


        A_a = A_0*X              # [m^2]
        r_a = (A_a/mt.pi)**0.5   # [m]

        U_x = m_L/(rho_p0*(A_0-A_a))     # [m/s] theorical x velocity
        U_r = m_L*R_s/(rho_p0*A_p*r_a)   # [m/s] theorical r velocity
        U_tan = U_x*np.tan(theta)        # [m/s] tangential velocity

        U = (U_x**2 + U_r**2 + U_tan**2)**0.5  # [m/s] mean velocity

        # Reference: "Atomization and sprays - Lefebvre."


        #%% SMD


        ParamSMD6 = 1.1938   # Tis parameter was added in this work.

        SMD6 = ParamSMD6*2.25*(Tens**0.25)*(mu_L**0.25)*(m_L**0.22)*(P_L**(-0.5))*(rho_A**(-0.25))

        SMD = SMD6

        # Reference: Quality of fuel atomization from small pressure-swirl atomizers - Milan Malý
        # (Bachelor's thesis)

        #%% Rotary atomizer (new spray, some equations must be replaced)

        # Now, the F_pin will be a input and the liquid pressure can be neglected

        D_0   = 154.39   # [um]
        S_0   = 0.5
        F_p0  = 0.021921
        T_p0r = 326.35
        a_smd = 400
        b_smd = -10.1
        c_smd = -600

        #F_pin = 88.2/3600  ;

        if j <= 200000:
            F_pin = 89.2/3600       # [-] solute fraction
            U_r = 45
        elif 200000 < j <= 325000 :
            F_pin = 89.2/3600
            U_r = 45
        else:
            F_pin = 89.2/3600
            U_r = 45


        T_p0  = T_R0 + 273.15
        S_in  = x_R

        # SMD unit: [m]

        SMD = (D_0 + a_smd*(F_pin - F_p0) + b_smd*(T_p0 - T_p0r) + c_smd*(S_in - S_0))*(10**(-6))/(MMD_SMD*1.45)

        m_L = F_pin


        #U_r = 45
        U_tan = 1
        U_x = U_r/2

        # U = (U_x**2 + U_r**2 + U_tan**2)**0.5


        #%% Text


        # print("El SMD es",SMD,"el flujo es",m_L*3600)
        # print("Los coeficientes son",SMD_exp/SMD,50/(m_L*3600))
        # print("los coeficientes de corrección fueron:",ParamSMD6,Param_m_L)




        Part = 1700   # Maximum number of iterations per batch of droplets

        # Initialization of variables

        V_ax  = np.zeros(Part)
        V_rad = np.zeros(Part)
        V_tan = np.zeros(Part)

        R_eq = np.zeros(Part)
        h_eq = np.zeros(Part)

        Mov_R = np.zeros(Part)
        Mov_Z = np.zeros(Part)

        M_R  = np.zeros(Part)
        M_Rw = np.zeros(Part)
        M_E  = np.zeros(Part)
        M_Ew = np.zeros(Part)

        M_Ewp = np.zeros(Part)
        M_wDL = np.zeros(Part)

        x_w  = np.zeros(Part)
        X_w  = np.zeros(Part)
        y_w  = np.zeros(Part)
        Y_w  = np.zeros(Part)

        Y_pt  = np.zeros(Part)
        Y_wDL = np.zeros(Part)

        T_R  = np.zeros(Part)

        # This vector corresponds to the initial values of the extract phase
        T_E_iter0 = 75

        T_E  = T_E_iter0*np.ones(Part)

        T_Ep = np.zeros(Part)

        parte3 = np.zeros(Part)
        d_drop = np.zeros(Part)

        V_rad[0] = U_r

        Mov_R[0] = 0
        Mov_Z[0] = 0

        #T_E0 = T_EC




        # Initial values

        m_air = 1710/3600 # kg/h to kg/s

        rho_air0 = 1.293*(273.15)/(273.15+T_E0)   # [kg/m^3]

        g = 9.8

        Vol_air = m_air/rho_air0  # [m^3/s] ... (1750(kg/h)/rho_air0(kg/m^3))/3600

        # Reference (Vol_air): "Modelling quality in spray drying - Kieviet."

        #m_air = (Vol_air*rho_air0)  # [kg/s] air flow
        D_eq  = 2                   # [m] Chamber diameter

        # Vel_g is the minimum velocity of the air in the chamber

        Vel_g = m_air/(rho_air0*np.pi*((D_eq/2)**2)) # [m/s]

        # Vel_gmax is the maximum velocity of the air in the chamber

        A_en_g   = np.pi*(0.2444**2 - 0.2024**2) # [m^2] area of air entrance
        Vel_gmax = m_air/(rho_air0*A_en_g)     # [m/s]

        # Initial values of variables

        V_ax[0]  = U_x     # [m/s] axial velocity of the droplet
        V_tan[0] = U_tan   # [m/s] tangetial velocity of the droplet

        R_eq[0] = 0  # [m] reference radial position of the nozzle
        h_eq[0] = 0  # [m] reference axial position of the nozzle

        m = rho_p0*np.pi*(1/6)*SMD**3 # [kg] Initial mass of one droplet

        # The "Gotas" value corresponds to the number of droplets in h_eu seconds
        # and remains constant.

        Gotas = (m_L/m)*h_eu  # [droplets/iteration]

        # Matrices para revisar cálculos

        T_R[0] = T_R0         # [°C] initial value of the reffinate

        M_R[0] = m_L*h_eu   # [kg/iteration] the mass in the first iteration

        x_w0 = 1-x_R      # [-] mass fraction of water

        M_Rw[0] = x_w0*m_L*h_eu  # [kg/iteration] the mass of water in the 1° iteration

        M_E[0]  = (m_air)*h_eu   # [kg/iteration] the mass of extract in the 1° iteration

        # Disturbance

        if j <= 175000:
            Y0 = 0.0021           # [-] Humidity
        elif 175000 < j <= 325000 :
            Y0 = 0.0021
        else:
            Y0 = 0.0021


        y_w0 = Y0/(1-Y0)     # [kg_W/kg_Extract]

        M_Ew[0] = y_w0*(m_air)*h_eu # [kg/iteration] the mass of water in the 1° iteration

        Y_w[0]  = M_Ew[0]/(M_E[0]-M_Ew[0])

        m_da = (1-y_w0)*m_air   # [kgDryAir/s] --- [kg_AireSeco/kg_AireHum]*[kg Aire húmedo/s]

        # Constants:

        m_ss = m_L*(1-x_w0)   # [kg_solid/s] solids flow
        m_ssi = m_ss*h_eu     # [kg_solid] solid mass in the first iteration
        m_ssg = m*(1-x_w0)    # [kg_solid/gota] solid per droplet

        MW = 18        # molecular weight of water
        MA = 28.96     # molecular weight of air

        R = 8314       # [J/kmol_k] [(Pa m^3)/(kmol K)]

        sigma = (5.67*10**(-8))/1000
        epsil = 0.96

        Cp_M  = 1.5     # [kJ/(kg K)]
        Cp_wl = 4.18    # [kJ/(kg K)]

        P = 101325      # [Pa]
        # T_Ref = 0       # °C It is defined before  ##### heater

        if 171 <= T_EC <= 181 :
            #U_Factor = 0.17*T_EC - 28.17
            U_Factor = 0.4
        elif T_EC < 171:
            U_Factor = 0.4
        else:
            #U_Factor = 2.6
            U_Factor = 0.4

        #U_h = 1.675/3600 # [kJ/(m^2 s)]
        U_h = 16.75*U_Factor/3600 # [kJ/(m^2 s)]



        d_drop[0] = SMD

        # Assuming that critical moisture and the diffusion coefficient
        # remains constant:

        X_cr = 0.54    # [kg_W/kg_solid] critical moisture
        #Diff = 4.1e-9  # [m^2/s]
        Diff = 2.5*5.9e-9  # [m^2/s]

        #%%

        #RemainW = m_L*h_eu
        RemainW = 0

        for i in range(0,Part-1):

            '''
            Este ciclo incluye la solución del sistema de ecuaciones que
            componen la velocidad de una sola gota en el spray.
            '''


            # Primero se halla la iteración previa que corresponde al recorrido

            if j == 0:


                #T_Ep[i] = T_E0
                T_Ep[i] = T_E_iter0
                Y_wDL[i] = Y0

                Vol_p0 = np.pi*((2/2)**2)*h_eu*V_ax[i]
                m_airp0 = P*Vol_p0*MA/(8.314*1000*(T_E0+273.15))

                M_wDL[i] = Y0*m_airp0

                k = i

                #T_Ep[k] = T_E0
                T_Ep[k] = T_E_iter0
                Y_wDL[k] = Y0


            else:

                # # # Variables temporales, se vuelven a definir al final del code

                ## Mov_VarTemp
                ## M_Ewp
                ## T_EpTemp


                Recorrido = Mov_Z[i]

                if Recorrido == 0:

                    # Ya no se pueden usar las variables anteriores por los índices
                    # de las ecuaciones nuevas

                    T_Ep[i] = T_E0
                    Y_wDL[i] = Y0

                    Vol_p0 = np.pi*((2/2)**2)*h_eu*V_ax[0]
                    m_airp0 = P*Vol_p0*MA/(8.314*1000*(T_E0+273.15))

                    M_wDL[i] = Y0*m_airp0

                    #M_wDL[i] = M_Ew0   # Antes tenía: M_Ewp[0]
                    #T_Ep[i]  = T_E0     # Antes tenía: T_EpTemp[0]

                    k = 0

                elif 0 <= Recorrido <= Mov_VarTemp[0]:

                    Y_wDL[i] = (Recorrido)*(Y_p[k]-Y_p[0])/(Mov_VarTemp[1]) + Y_p[0]
                    M_wDL[i] = (Recorrido)*(M_Ewp[k]-M_Ewp[0])/(Mov_VarTemp[1]) + M_Ewp[0]
                    T_Ep[i] = (Recorrido)*(T_EpTemp[k]-T_EpTemp[0])/(Mov_VarTemp[1]) + T_EpTemp[0]

                    k = 0

                elif Mov_VarTemp[0] <= Recorrido <= Mov_VarTemp[-1]:


                    # CÓDIGO CORREGIDO Y ROBUSTO
                    denominador = Mov_VarTemp[i] - Mov_VarTemp[i-1]

                    if abs(denominador) < 1e-9:
                        # Si el denominador es cero, no podemos interpolar.
                        # Asignamos el valor anterior como una aproximación segura.
                        Y_wDL[i] = Y_p[i-1]
                        M_wDL[i] = M_Ewp[i-1]
                        T_Ep[i]  = T_EpTemp[i-1]

                    elif Mov_VarTemp[i-2] <= Recorrido <= Mov_VarTemp[i-1]:

                        Y_wDL[i] = (Recorrido-Mov_VarTemp[i-2])*(Y_p[i-1]-Y_p[i-2])/(Mov_VarTemp[i-1]-Mov_VarTemp[i-2]) + Y_p[i-2]
                        M_wDL[i] = (Recorrido-Mov_VarTemp[i-2])*(M_Ewp[i-1]-M_Ewp[i-2])/(Mov_VarTemp[i-1]-Mov_VarTemp[i-2]) + M_Ewp[i-2]
                        T_Ep[i] = (Recorrido-Mov_VarTemp[i-2])*(T_EpTemp[i-1]-T_EpTemp[i-2])/(Mov_VarTemp[i-1]-Mov_VarTemp[i-2]) + T_EpTemp[i-2]

                        k = i-1

                        #k = i

                        #break

                    elif Mov_VarTemp[i] <= Recorrido <= Mov_VarTemp[i+1]:

                        Y_wDL[i] = (Recorrido-Mov_VarTemp[i])*(Y_p[i+1]-Y_p[i])/(Mov_VarTemp[i+1]-Mov_VarTemp[i]) + Y_p[i]
                        M_wDL[i] = (Recorrido-Mov_VarTemp[i])*(M_Ewp[i+1]-M_Ewp[i])/(Mov_VarTemp[i+1]-Mov_VarTemp[i]) + M_Ewp[i]
                        T_Ep[i] = (Recorrido-Mov_VarTemp[i])*(T_EpTemp[i+1]-T_EpTemp[i])/(Mov_VarTemp[i+1]-Mov_VarTemp[i]) + T_EpTemp[i]

                        k = i+1

                        #k = i+2

                    elif Mov_VarTemp[i-3] <= Recorrido <= Mov_VarTemp[i-2]:

                        Y_wDL[i] = (Recorrido-Mov_VarTemp[i-3])*(Y_p[i-2]-Y_p[i-3])/(Mov_VarTemp[i-2]-Mov_VarTemp[i-3]) + Y_p[i-3]
                        M_wDL[i] = (Recorrido-Mov_VarTemp[i-3])*(M_Ewp[i-2]-M_Ewp[i-3])/(Mov_VarTemp[i-2]-Mov_VarTemp[i-3]) + M_Ewp[i-3]
                        T_Ep[i] = (Recorrido-Mov_VarTemp[i-3])*(T_EpTemp[i-2]-T_EpTemp[i-3])/(Mov_VarTemp[i-2]-Mov_VarTemp[i-3]) + T_EpTemp[i-3]

                        k = i-2

                        #k = i-1

                    elif Mov_VarTemp[i+1] <= Recorrido <= Mov_VarTemp[i+2]:

                        Y_wDL[i] = (Recorrido-Mov_VarTemp[i+1])*(Y_p[i+2]-Y_p[i+1])/(Mov_VarTemp[i+2]-Mov_VarTemp[i+1]) + Y_p[i+1]
                        M_wDL[i] = (Recorrido-Mov_VarTemp[i+1])*(M_Ewp[i+2]-M_Ewp[i+1])/(Mov_VarTemp[i+2]-Mov_VarTemp[i+1]) + M_Ewp[i+1]
                        T_Ep[i] = (Recorrido-Mov_VarTemp[i+1])*(T_EpTemp[i+2]-T_EpTemp[i+1])/(Mov_VarTemp[i+2]-Mov_VarTemp[i+1]) + T_EpTemp[i+1]

                        k = i+2

                        #k = i+3



                    else:
                        #print("Sí se usa")
                        u = Recorrido + 0.6
                        a = np.where(np.logical_and(np.greater_equal(Mov_VarTemp,Recorrido),np.less_equal(Mov_VarTemp,u)))[0][0]

                        if Recorrido <= Mov_VarTemp[-1]:
                            k = a

                        else:
                            k = len(Mov_VarTemp)

                        # En los siguientes condicionales se hallan las humedades y temperaturas previas

                        if k == 0: # Interpolación con datos iniciales de humedad y temperatura

                            Y_wDL[i] = (Recorrido)*(Y_p[k]-Y_p[0])/(Mov_VarTemp[1]) + Y_p[0]
                            M_wDL[i] = (Recorrido)*(M_Ewp[k]-M_Ewp[0])/(Mov_VarTemp[1]) + M_Ewp[0]
                            T_Ep[i] = (Recorrido)*(T_EpTemp[k]-T_EpTemp[0])/(Mov_VarTemp[1]) + T_EpTemp[0]

                            k = 0

                        elif 0 < k < len(Mov_VarTemp): # Interpolación lineal para humedad y temperatura previa

                            #print("Este sale")
                            Y_wDL[i] = (Recorrido-Mov_VarTemp[k-1])*(Y_p[k]-Y_p[k-1])/(Mov_VarTemp[k]-Mov_VarTemp[k-1]) + Y_p[k-1]
                            M_wDL[i] = (Recorrido-Mov_VarTemp[k-1])*(M_Ewp[k]-M_Ewp[k-1])/(Mov_VarTemp[k]-Mov_VarTemp[k-1]) + M_Ewp[k-1]
                            T_Ep[i] = (Recorrido-Mov_VarTemp[k-1])*(T_EpTemp[k]-T_EpTemp[k-1])/(Mov_VarTemp[k]-Mov_VarTemp[k-1]) + T_EpTemp[k-1]

                            k = k

                        elif k == len(Mov_VarTemp):  # Extrapolación lineal para humedad y temperatura previa

                            Y_wDL[i] = (Recorrido-Mov_VarTemp[-2])*(Y_p[-1]-Y_p[-2])/(Mov_VarTemp[-1]-Mov_VarTemp[-2]) + Y_p[-2]
                            M_wDL[i] = (Recorrido-Mov_VarTemp[-2])*(M_Ewp[-1]-M_Ewp[-2])/(Mov_VarTemp[-1]-Mov_VarTemp[-2]) + M_Ewp[-2]
                            T_Ep[i] = (Recorrido-Mov_VarTemp[-2])*(T_EpTemp[-1]-T_EpTemp[-2])/(Mov_VarTemp[-1]-Mov_VarTemp[-2]) + T_EpTemp[-2]

                else:

                    Y_wDL[i] = (Recorrido-Mov_VarTemp[-2])*(Y_p[-1]-Y_p[-2])/(Mov_VarTemp[-1]-Mov_VarTemp[-2]) + Y_p[-2]
                    M_wDL[i] = (Recorrido-Mov_VarTemp[-2])*(M_Ewp[-1]-M_Ewp[-2])/(Mov_VarTemp[-1]-Mov_VarTemp[-2]) + M_Ewp[-2]
                    T_Ep[i] = (Recorrido-Mov_VarTemp[-2])*(T_EpTemp[-1]-T_EpTemp[-2])/(Mov_VarTemp[-1]-Mov_VarTemp[-2]) + T_EpTemp[-2]

                    # El próximo renglón es de emergencia
                    # Con gas fluyendo, no será requerido

                    k = len(Mov_VarTemp)

                # Se halla el índice correspondiente:

                # a = np.where(Mov_VarTemp>=Recorrido)[0].tolist()
                # a = np.where(Mov_VarTemp>=Recorrido and u>=Mov_VarTemp)[0].tolist()


            # if Mov_Z[i] < 1.77:
            #     D_eq = 2.2
            # else:
            #     D_eq = 2*(1.9-(Mov_Z[i]-1.77))*np.tan(30*np.pi/180)

            # Los nuevos cálculos de X_w y Y_w se hacen con los solventes sin la necesidad de iterar

            #RemainW = RemainW -

            #x_w[i] = M_Rw[i]/(RemainW)
            #X_w[i] = x_w[i]/(1-x_w[i])
            X_w[i] = M_Rw[i]/m_ssi  # Es posible porque m_ssi:[kgSólido(SolventeR)/iter]
            x_w[i] = M_Rw[i]/(m_ssi + M_Rw[i])
            # Y_w[i] = M_Ew[i]/(m_da*h_eu) # m_da:[kgAireSeco/s]*[s/iter]

            Delta_L = h_eu*V_ax[i]

            if Mov_Z[i] < 2.3:
                D_eqp = 2
            else:
                #D_eqp = 2*(1.7-(Mov_Z[i]-1.77))*np.tan(30*np.pi/180)
                D_eqp = (np.tan(20*np.pi/180))*(2*np.tan(70*np.pi/180)-(2.3-Mov_Z[i]))

            # if Mov_Z[i] < 1.77:
            #     D_eq = 2
            # else:
            #     D_eq = 2*(1.7-(Mov_Z[i]-1.77))*np.tan(30*np.pi/180)


            Vol = np.pi*((D_eq/2)**2)*Delta_L     # [m^3]

            y_w[i] = Y_wDL[i]/(1+Y_wDL[i])

            # Propiedades del líquido y del aire

            rho_M = (1.635 - 0.0026*T_R[i] + 2*10**(-5)*T_R[i]**2)
            rho_W = (1.0020825 - 1.14*10**(-4)*T_R[i] - 3.325*10**(-6)*T_R[i]**2)*1000
            rho_p = 100*1000/(100*(1-x_w[i])*(1/rho_M - 1/(rho_W/1000)) + 100/(rho_W/1000))

            # Cp_M = 1.5    # [kJ/(kg K)]
            # Cp_wl = 4.18  # [kJ/(kg K)]

            lamb_v = (3.15e3 - (T_R[i]+273.15)*2.38)   # [kJ/(kg)] Art. 3, nota: Art. 6 también lo tiene pero erróneo. Se hace la corrección

            #P = 101325   # [Pa]

            mu_air  = 1.72*10**(-5) + 4.568*(10**(-8))*T_Ep[i]     # [kg Aire/(m s)] Viscosidad dinámica del Aire. Artículo 6
            rho_Air = 1.293*(273.15)/(273.15+T_Ep[i])              # [kg Aire/m^3] Densidad del Aire.
            D_air   = (2.302*(1e-5)*0.98*(10**5)/P)*( ((T_Ep[i]+T_R[i])/2 + 273.15 )/256 )**1.81  # [m^2/s] Difusividad del Vapor de agua en el Aire. ARTÍCULO 3

            k_d    = 1.731*( 0.014 + 4.296*(T_Ep[i])*(1e-5))/1000   # [kW/(m K)] Conductividad del aire. Art 6
            Cp_a   = (969.542 + 6.801*(T_Ep[i]+273.15)*1e-2 + 16.569*((T_Ep[i]+273.15)**2)*1e-5 - 67.828*((T_Ep[i]+273.15)**3)*1e-9)/1000    # [kJ/(kg K)] La T es en K. Art 6
            # [kJ/(kg K)]Capacidad calorífica del vapor de agua. Art 6

            # El cp del vapor de agua fue corregido, había error por conversión. En el artículo 6 tienen malas las unidades
            Cp_vap = (1.883 - 1.674*(T_Ep[i])*(1e-4)+ 8.4390*((T_Ep[i])**2)*(1e-7) - 2.6970*((T_Ep[i])**3)*1e-10)  # [kJ/(kg K)]Capacidad calorífica del vapor de agua. Art 6

            P_sat = (101325/760)*10**(7.95581-(1668.210/(T_R[i]+228)))  # [Pa]. Presión de vapor saturado del Agua

            #Y_wmol = Y_wDL[i]*(MA/MW)
            y_wmol  = Y_wDL[i]*(MA/MW)/(1+Y_wDL[i]*(MA/MW))

            #y_wmol = (M_Ew[i]/MW)/(M_Ew[i]/MW + m_da*h_eu/MA)  # [mol_W/mol_AireHúmedo]
            P_v   = P*y_wmol                                    # [Pa] Presión parcial del agua

            # Integraciones y entalpías

            #T_Ref = 0  # °C
            # El Cp_aire está en [kJ/(kg K)] La T es en K. Art 6

            C_EntA1 = (969.542/1000)*(T_Ep[i] - T_Ref)
            C_EntA2 = ((6.801*1e-2)/1000)*( ((T_Ep[i]+273.15)**2)/2 - ((T_Ref+273.15)**2)/2 )
            C_EntA3 = ((16.569*1e-5)/1000)*( ((T_Ep[i]+273.15)**3)/3 - ((T_Ref+273.15)**3)/3 )
            C_EntA4 = ((-67.828*1e-9)/1000)*( ((T_Ep[i]+273.15)**4)/4 - ((T_Ref+273.15)**4)/4 )

            # El Cp_vaporW está en [kJ/(kg K)] Capacidad calorífica del vapor de agua. Art 6

            C_EntV1 = (1.883)*(T_Ep[i] - T_Ref)
            C_EntV2 = ((-1.674*1e-4))*( ((T_Ep[i])**2)/2 - ((T_Ref)**2)/2 )
            C_EntV3 = ((8.4390*1e-7))*( ((T_Ep[i])**3)/3 - ((T_Ref)**3)/3 )
            C_EntV4 = ((-2.6970*1e-10))*( ((T_Ep[i])**4)/4 - ((T_Ref)**4)/4 )

            H_a  = C_EntA1 + C_EntA2 + C_EntA3 + C_EntA4  # [kJ/kg]
            H_vw = C_EntV1 + C_EntV2 + C_EntV3 + C_EntV4  # [kJ/kg]

            # Números adimensionales

            # Tamaño de gota (requerido para Re)

            X0 = x_w0/(1-x_w0)

            # Pausa

            if X_w[i] > X_cr:

                f = 0
                d_drop[i] = (SMD**3 - 6*m_ssg*(X0 - X_w[i])/(np.pi*rho_W))**(1/3)

            else:

                #if T_R[i] < 100:
                if P_sat < P:

                    aw = P_v/P_sat
                    Weq = 0.05*np.exp(-99.27/(T_R[i]+273.15))
                    Keq = 0.65*np.exp(144.57/(T_R[i]+273.15))
                    Ceq = 0.04*np.exp(1257.14/(T_R[i]+273.15))
                    ConcReq = Ceq*Keq*Weq*aw/((1-Keq*aw)*(1- Keq*aw + Ceq*Keq*aw))
                    f = ((X_w[i]-ConcReq)/(X_cr-ConcReq))**(-1/3) -1

                    d_drop[i] = (SMD**3 - 6*m_ssg*(X0 - X_cr)/(np.pi*rho_W))**(1/3)

                else:

                    rho_W100 = (1.0020825 - 1.14*10**(-4)*100 - 3.325*10**(-6)*100**2)*1000
                    d_drop[i] = (SMD**3 - 6*m_ssg*(X0 - X_cr)/(np.pi*rho_W100))**(1/3)


            Vel_g = 0*m_air/(rho_Air*np.pi*((2/2)**2))
            ## Vel_gmax = 1.5

            #V_gz = 1


            #Coef_velz = 7.376*0.45
            Coef_velz = 7.376*0.45
            V_gz = Coef_velz*(((D_eq - Mov_R[i])/D_eq)**2.5)

            #Coef_velt = 0.7
            Coef_velt = 0.7
            V_gt = Coef_velt*(((D_eq - Mov_R[i])/D_eq)**0.5)

            if 0<=Mov_R[i]<=0.15:
                if 0<=Mov_Z[i]<=0.1:
                    V_gr = -5.19
                else:
                    V_gr = 0
            else:
                V_gr = 0

            # <>

            V_rel = ((V_ax[i] - V_gz)**2 + (V_tan[i] - V_gt)**2 +(V_rad[i] - V_gr)**2)**0.5
            Re = rho_Air*d_drop[i]*np.abs(V_rel)/mu_air

            Sc = mu_air/(rho_Air*D_air)

            B = Cp_vap*(T_Ep[i]-T_R[i])/lamb_v
            Sh = (2 + 0.6*(Re**0.5)*Sc**(1/3))/((1+B)**0.7)

            Pr = mu_air*Cp_a/k_d

            # Coeficientes

            kmass = 0.4*Sh*D_air/d_drop[i]
            # CÓDIGO CORREGIDO PARA h_rad
            # Convertimos las temperaturas a Kelvin
            T_Ep_K = T_Ep[i] + 273.15
            T_R_K = T_R[i] + 273.15

            # Usamos la fórmula matemáticamente estable que evita la división
            h_rad = sigma * epsil * (T_Ep_K + T_R_K) * (T_Ep_K**2 + T_R_K**2) / 1000


            h_conv = (k_d/d_drop[i])*(2 + 0.6*(Re**0.5)*(Pr**1/3))/((1+B)**0.7)

            h_v = (h_rad + h_conv)  # [kW/(m^2 K)]

            #if T_R[i] < 100:

            if P_sat < P:

                parte1 = P*MW/(R*((T_R[i]+T_Ep[i])/2 + 273.15))
                parte2 = 2*Diff/(d_drop[i]*(f + 2*Diff/(kmass*d_drop[i])))
                parte3[i] = np.log((P - P_v)/(P - P_sat))

                mv_flux = parte1*parte2*parte3[i]  # [kg Vapor de Agua/(m^2 s)] Flux de vapor de Agua que se transifere

                m_evap = mv_flux*Gotas*np.pi*(d_drop[i]**2)  # [kg Agua/(iter s)]
            else:
                #print("s")
                if X_w[i] > 0:
                    m_evap = Gotas*np.pi*(d_drop[i]**2)*h_v*(T_Ep[i]-T_R[i])/lamb_v
                else:
                    m_evap = 0
                    #X_w[i] = 0

            if m_evap <= 0:
                m_evap = 0


            # Fuerzas F_dr (Fuerza droplet radial), F_ar (Fuerza de masa añadida radial)
            # y Sum_Frad (suma de fuerzas radiales).

            #X_d = 1
            # <>

            m_gota = m_ssg + M_Rw[i]/Gotas

            # Coeficientes de arrastre de 2014 Muzammil

            # CÓDIGO CORREGIDO PARA C_drag
            if Re < 1e-9: # Usamos un número pequeño para evitar problemas con decimales
                C_drag = 0 # Si Re es cero, no hay arrastre
            else:
                # Si Re no es cero, procedemos con el cálculo original
                if X_w[i] > X_cr:
                    if Re > 80:
                        C_drag = 0.271*Re**0.217   # verificar
                    else:
                        C_drag = 27*Re**(-0.84)
                else:
                    if Re <= 0.1:
                        a_1 = 0
                        a_2 = 24
                        a_3 = 0

                    elif Re <= 1:
                        a_1 = 3.69
                        a_2 = 22.73
                        a_3 = 0.0903

                    elif Re <= 10:
                        a_1 = 1.222
                        a_2 = 29.167
                        a_3 = -3.889

                    elif Re <= 100:
                        a_1 = 0.6167
                        a_2 = 46.5
                        a_3 = -116.667

                    elif Re <= 1000:
                        a_1 = 0.3644
                        a_2 = 98.33
                        a_3 = -2778

                    elif Re <= 5000:
                        a_1 = 0.357
                        a_2 = 148.62
                        a_3 = -4.75e4

                    elif Re <= 10000:
                        a_1 = 0.46
                        a_2 = -490.546
                        a_3 = 57.87e4

                    else:
                        a_1 = 0.5191
                        a_2 = -1662.5
                        a_3 = 5.4167e6

                    C_drag = a_1 + a_2/Re + a_3/(Re**2)


            F_dt = (np.pi/8)*rho_Air*(d_drop[i]**2)*C_drag*(np.abs(V_gt - V_tan[i]))*(V_gt - V_tan[i])
            F_dr = (np.pi/8)*rho_Air*(d_drop[i]**2)*C_drag*(np.abs(V_gr - V_rad[i]))*(V_gr - V_rad[i])
            alfa = (np.pi/12)*rho_Air*(d_drop[i]**3)


            # Fuerzas F_dz (Fuerza droplet axial), F_az (Fuerza de masa añadida axial)
            # y Sum_Frad (suma de fuerzas axiales).

            F_dz = (np.pi/8)*rho_Air*(d_drop[i]**2)*C_drag*(np.abs(V_gz - V_ax[i]))*(V_gz - V_ax[i])
            F_b  = -(np.pi/8)*rho_Air*(d_drop[i]**3)*g

            Sum_Fax = F_dz + F_b


            # Ecuaciones diferenciales


            # Ecuación diferencial dV_tan/dt y método Euler:

            dV_tan = F_dt/(m_gota + alfa)
            V_rad[i+1]  = V_tan[i] + h_eu*dV_tan

            # Ecuación diferencial dV_rad/dt y método Euler:

            dV_rad = F_dr/(m_gota + alfa)
            V_rad[i+1]  = V_rad[i] + h_eu*dV_rad

            # Ecuación diferencial dV_ax/dt y método Euler:

            dV_ax = (Sum_Fax/m_gota + g)/(1 + alfa/m_gota)
            V_ax[i+1] = V_ax[i] + h_eu*dV_ax

            #V_gz[i+1] = V_gz[i]

            # Posiciones

            # Estas ecuaciones fueron excluídas porque se actualizaba
            # el índice y la posición 0 se movía.
            #R_eq[i] = R_eq[0] + h_eu*V_rad[i]
            #h_eq[i] = h_eq[0] + h_eu*V_ax[i]

            Mov_R[i+1] = Mov_R[i] + h_eu*V_rad[i]
            Mov_Z[i+1] = Mov_Z[i] + h_eu*V_ax[i]


            # Términos y herramientas para las ec diferenciales de balances
            # Las distancias son requeridas para los Delta_L y las condiciones iniciales

                    # <>

            # Partes de las ecuaciones masa (corregir los últimos renglones que son de energía)

            # El siguiente coeficiente está en [1/s]

            if Mov_Z[i] < 2.3:
                D_eqp = 2
            else:
                #D_eqp = 2*(1.7-(Mov_Z[i]-1.77))*np.tan(30*np.pi/180)
                D_eqp = (np.tan(20*np.pi/180))*(2*np.tan(70*np.pi/180)-(2.3-Mov_Z[i]))

            rho_Airk = 1.293*(273.15)/(273.15+T_EpTemp[k])

            f_air_i = rho_Air*Vel_g*np.pi*((D_eq/2)**2)
            f_air_k = rho_Airk*Vel_g*np.pi*((D_eq/2)**2)

            # Partes de las ecuaciones de energía

            C_EntA1p = (969.542/1000)*(T_EpTemp[k] - T_Ref)
            C_EntA2p = ((6.801*1e-2)/1000)*( ((T_EpTemp[k]+273.15)**2)/2 - ((T_Ref+273.15)**2)/2 )
            C_EntA3p = ((16.569*1e-5)/1000)*( ((T_EpTemp[k]+273.15)**3)/3 - ((T_Ref+273.15)**3)/3 )
            C_EntA4p = ((-67.828*1e-9)/1000)*( ((T_EpTemp[k]+273.15)**4)/4 - ((T_Ref+273.15)**4)/4 )

            C_EntV1p = (1.883)*(T_EpTemp[k] - T_Ref)
            C_EntV2p = ((-1.674*1e-4))*( ((T_EpTemp[k])**2)/2 - ((T_Ref)**2)/2 )
            C_EntV3p = ((8.4390*1e-7))*( ((T_EpTemp[k])**3)/3 - ((T_Ref)**3)/3 )
            C_EntV4p = ((-2.6970*1e-10))*( ((T_EpTemp[k])**4)/4 - ((T_Ref)**4)/4 )

            H_app  = C_EntA1p + C_EntA2p + C_EntA3p + C_EntA4p
            H_vwpp = C_EntV1p + C_EntV2p + C_EntV3p + C_EntV4p

            C_Rv1 = (1.883)*(T_Ep[i] - T_R[i])
            C_Rv2 = ((-1.674*1e-4))*( ((T_Ep[i])**2)/2 - ((T_R[i])**2)/2 )
            C_Rv3 = ((8.4390*1e-7))*( ((T_Ep[i])**3)/3 - ((T_R[i])**3)/3 )
            C_Rv4 = ((-2.6970*1e-10))*( ((T_Ep[i])**4)/4 - ((T_R[i])**4)/4 )

            H_wvevap = C_Rv1 + C_Rv2 + C_Rv3 + C_Rv4

            # En este caso se usan las tandas previas con iteración próxima
            # Es decir, pp (previa, próxima)

            lamb_v0 = (3.15e3 - (T_Ref + 273.15)*2.38)

            H_p = H_app + (H_vwpp + lamb_v0)*Y_p[k]  # [kJ/kg]
            H   = H_a  + (H_vw + lamb_v0)*Y_wDL[i]       # [kJ/kg]

            # Differential equations related to the mass balance:

            dM_Rwdt = -m_evap
            dM_Ewdt    = (f_air_k*Y_p[k] - f_air_i*Y_wDL[i] + m_evap )

            M_Rw[i+1] = M_Rw[i] + h_eu*( dM_Rwdt )

            RemainW = RemainW + m_evap

            m_airp = P*Vol*MA/(8.314*1000*(T_Ep[i]+273.15))
            M_Ew[i] = Y_wDL[i]*m_airp + h_eu*( dM_Ewdt )

            ######Y_pt[i] = M_Ew[i]/m_airp


            # Differential equations related to the energy balance:

            denTR = M_Rw[i]*Cp_wl + (m_ssi)*Cp_M

            Cv_air = Cp_a - 8.314/MA
            Cv_wv  = Cp_vap - 8.314/MW

            #dCv_wvdT = 0.005 #( - 1.674*(1e-4)+ 2*8.4390*((T_Ep[i])**1)*(1e-7) - 3*2.6970*((T_Ep[i])**2)*1e-10)
            #dCv_airdT = ( 6.801*1e-2 + 2*16.569*((T_Ep[i]+273.15)**1)*1e-5 - 3*67.828*((T_Ep[i]+273.15)**2)*1e-9)/1000


            dCv_wvdT = (2.558e-12)*(T_Ep[i]**3) + (-4.681e-09)*(T_Ep[i]**2) + (2.615e-06)*T_Ep[i] + 0.0001957
            dCv_airdT = (9.785e-06)*(T_Ep[i]**(0.5526)) -1.634e-05


            denTE = ((m_airp*(T_Ep[i]+273.15)*dCv_airdT + Y_wDL[i]*m_airp*(T_Ep[i]+273.15)*dCv_wvdT )+(m_airp*Cv_air + Y_wDL[i]*m_airp*Cv_wv)) # + np.abs(np.pi*((R_equipo**2)-(r_eq**2))*Delta_L*rho_wall*Cp_wall)

            NumTR = -m_evap*(Cp_wl*(T_R[i]-T_Ref) + lamb_v) + Gotas*np.pi*(d_drop[i]**2)*h_v*(T_Ep[i]-T_R[i]) - Cp_M*(T_R[i]+273.15)*dM_Rwdt

            T_Edif = Cv_wv*(T_Ep[i]+273.15)*dM_Ewdt

            NumTE = f_air_k*H_p - f_air_i*H + m_evap*H_wvevap - Gotas*np.pi*(d_drop[i]**2)*h_v*(T_Ep[i]-T_R[i]) - (np.pi*D_eqp*Delta_L)*U_h*(T_Ep[i]-25) - T_Edif

            dT_Rdt = NumTR/denTR
            dT_Edt = NumTE/denTE

            # Euler solution:

            T_R[i+1]  = T_R[i] + h_eu*( dT_Rdt )

            T_E[i] = T_Ep[i] + h_eu*( dT_Edt )

            Y_pt[i] = M_Ew[i]/(P*Vol*MA/(8.314*1000*(T_E[i]+273.15)))

            if Mov_Z[i] > 3.5:

                # print()
                # print("Se alcanzó la altura del equipo")
                # print("La última iteración fue la",i)
                # print("El diferencial de tiempo para las iteraciones es de:",h_eu,"s")
                # print("El tiempo de residencia es de:",h_eu*i,"s")

                if j == 0:

                    print()
                    print("Se alcanzó la altura del equipo")
                    print("La última iteración fue la",i)
                    print("El diferencial de tiempo para las iteraciones es de:",h_eu,"s")
                    print("El tiempo de residencia es de:",h_eu*i,"s")

                if j == (T-1):

                    print()
                    print("Se alcanzó la altura del equipo")
                    print("La última iteración fue la",i)
                    print("El diferencial de tiempo para las iteraciones es de:",h_eu,"s")
                    print("El tiempo de residencia es de:",h_eu*i,"s")

                T_Eg = T_Ep[0:i]

                T_EpTemp = T_E[0:i]




                #f_param = 0.75

                #if j % 250 == 0:  # standard
                if j % 250 == 0:

                    f_balanc = ( (Y0/(1+Y0))*m_air + m_L*(1-x_R) - (1-Y0)*m_air*Y_wDL[i] - (x_R*m_L)*X_w[i] )*3600# [kg/h]

                    if abs(f_balanc) > 1 :
                        if 0.6 < f_param < 1.4:
                            if f_balanc < 0:
                                f_param = f_param + 0.001
                            else:
                                f_param = f_param - 0.001



                #Vel_gF = m_air/(rho_Air*np.pi*((2.4/2)**2))
                Vel_gF = m_air/(rho_air0*np.pi*((2/2)**2))

                #Mov_VarTemp = Mov_Z[0:i] + 1.575*Vel_gF*h_eu
                Mov_VarTemp = Mov_Z[0:i] + f_param*Vel_gF*h_eu

                #+ Vel_g*h_eu*0.55 #+ 2*Vel_g*h_eu
                #Y_p = Y_w[0:i]

                M_Ewp = M_Ew[0:i]

                Y_graf = Y_wDL[0:i]

                Vel_graf = V_ax[0:i]

                # # Gráficas

                #Vel_gv = Vel_g*np.ones(i)    # gv: gas vector

                # if j == (T-1):
                #     T_Egraf = T_E[0:i]
                #     Mov_Epgraf = Mov_Z[0:i] + Vel_g*h_eu

                # if j < T:
                #     T_EpTemp = T_E[0:i]
                #     Mov_VarTemp = Mov_Z[0:i] + Vel_g*h_eu
                #     Y_p = Y_w[0:i]
                # else:
                #     T_EpGraf1 = T_EpTemp
                #     Mov_EpGraf = Mov_VarTemp


                # <>

                # Cálculo de los Y previos completos

                #DeltasL = h_eu*V_ax

                # Vol_p = np.pi*((2.2/2)**2)*h_eu*V_ax[0:i]
                # m_airpT = P*Vol_p*MA/(8.314*1000*(T_EpTemp+273.15))

                # Y_p = M_Ewp/m_airpT

                Y_p = Y_pt[0:i]


                if j % 1000 == 0:

                    #NI = int(j/1000)
                    #T_Ef = T_Eg[-1]

                    T_ECT.append(T_EC)
                    T_Eout.append(T_Eg[-1])
                    Y_Eout.append(Y_graf[-1])
                    Z_Mout.append(X_w[i])

                    R_time.append(h_eu*i)

                    P_inMV.append(P_in)
                    m_LMV.append(m_L)
                    #T_Eout.append(T_Ef)
                    #np.append(T_Eout,T_Ef)
                    # archivo = open("Texto.txt","w")
                    # archivo.write(T_Ef)
                    # archivo.close()


                if j == 1000: #17000:

                    Bef = (i,d_drop[0:i],Mov_Z[0:i],T_Eg,Y_graf,T_R[0:i],Mov_R,M_Ew,X_w[0:i], mv_flux)

                if j == 2000: #18000:

                    Bef1 = (i,d_drop[0:i],Mov_Z[0:i],T_Eg,Y_graf,T_R[0:i],Mov_R,M_Ew,X_w[0:i], mv_flux)

                if j == 3000: #19000:

                    Bef2 = (i,d_drop[0:i],Mov_Z[0:i],T_Eg,Y_graf,T_R[0:i],Mov_R,M_Ew,X_w[0:i], mv_flux)

                if j == 4000: #20000:

                    Bef3 = (i,d_drop[0:i],Mov_Z[0:i],T_Eg,Y_graf,T_R[0:i],Mov_R,M_Ew,X_w[0:i], mv_flux)

                if j == 15000: #21000:

                    Bef4 = (i,d_drop[0:i],Mov_Z[0:i],T_Eg,Y_graf,T_R[0:i],Mov_R,M_Ew,X_w[0:i], mv_flux)

                if j == 20000: #22000:

                    Bef5 = (i,d_drop[0:i],Mov_Z[0:i],T_Eg,Y_graf,T_R[0:i],Mov_R,M_Ew,X_w[0:i], mv_flux)

                if j == 30000: #28000:

                    Bef6 = (i,d_drop[0:i],Mov_Z[0:i],T_Eg,Y_graf,T_R[0:i],Mov_R,M_Ew,X_w[0:i], mv_flux)

                break

    return (parte3[0:i], d_drop[0:i], i, Mov_Z[0:i], T_Eg, Y_graf, T_R[0:i], V_gz, Mov_R, M_Ew,  # llega al 9
            Vel_graf, X_w[0:i], Y_p, mv_flux, Vel_g, Vel_gF,
            Bef,Bef1,Bef2,Bef3,Bef4,Bef5,Bef6,T_Eout,Y_Eout,T_ECT,Z_Mout,
            P_inMV,m_LMV, RemainW, f_param, f_balanc, R_time)

# Pausa

Mov_VarTemp_1 = np.zeros(1500)
T_EpTemp_1    = np.zeros(1500)
Y_p_1         = np.zeros(1500)
M_wDL_1       = np.zeros(1500)

Vector = Spray(T,Mov_VarTemp_1,T_EpTemp_1,Y_p_1)

i = Vector[2]
parte3 = Vector[1]

Mov_Z = Vector[3]
T_E = Vector[4]
Y_w = Vector[5]
T_R = Vector[6]

Vel_g = Vector[7]

Mov_R = Vector[8][0:Vector[2]]

Mov_ZR = Mov_Z
Mov_ZE = Mov_Z

Bef = Vector[16]
Bef1 = Vector[17]
Bef2 = Vector[18]
Bef3 = Vector[19]
Bef4 = Vector[20]
Bef5 = Vector[21]
Bef6 = Vector[22]

#%%

#TheLast = Vector[-1][0:Vector[2]]
#%%

# print()
# print("Se alcanzó la altura del equipo")
# print("La última iteración fue la",i)
# print("El diferencial de tiempo para las iteraciones es de:",h_eu,"s")
# print("El tiempo de residencia es de:",h_eu*i,"s")

elapsed = (time.process_time() - start)

print()
print(elapsed, "seconds")

#%%

x_expz = [0,0.2,0.2,0.6,1.0,1.40,2.6]
y_expr = [0,0,0.29,0.29,0.29,0.29,0.0]

fig, ax = plt.subplots(1, figsize=(5,7))

ax.plot(Mov_R, Mov_Z,linewidth=0.5,color='k') # Datos de los ejes y color
ax.scatter(y_expr,x_expz, color='k',s=5)

ax.set(xlabel="Radial distance from nozzle [m]",
        ylabel="Axial distance from nozzle [m]")
        #title='Recorrido de la partícula')

ax.grid()
ax.grid(color='#CCCCCC', linestyle='dotted', linewidth=0.5)

#plt.axes().set_aspect('equal')  # Hace que las celdas tengan las mismas proporciones

for axis in ['top','bottom','left','right']:
    ax.spines[axis].set_linewidth(0.5) # This line is now indented


#ax.set_xlim([0, (mt.ceil(Mov_R[-1]))/2 ])
#ax.set_ylim([0, mt.ceil(Mov_Z[-1])])
plt.gca().invert_yaxis()                       # Invierte el eje y

# Límites de los ejes:

plt.yticks(np.arange(0, mt.ceil(Mov_Z[-1]), 0.25))
#plt.xticks(np.arange(0, mt.ceil(Mov_R[-1])/2, 0.05))
plt.xticks(np.arange(0, 0.7, 0.05))
#plt.axis('scaled')

fig.savefig("Figure2.pdf", dpi=600)
plt.show()



#%%

# x_exp = [0,0.2,0.6,1.00,1.40,2.3]
# y_exp = [195,143.675, 113.233, 121.858, 113.417,113]

x_exp = [0,0.2,0.6,1.0,1.40,2.3]
y_exp = [195,175, 113.233, 121.858, 113.417,113]

fig, ax = plt.subplots(1, figsize=(10,5))





# ax.plot(Bef[2],Bef[5],linewidth=0.5,color='b')
# ax.plot(Bef[2],Bef[3],linewidth=0.5,color='r')

# ax.plot(Bef1[2],Bef1[5],linewidth=0.5,color='b')
# ax.plot(Bef1[2],Bef1[3],linewidth=0.5,color='r')

# ax.plot(Bef2[2],Bef2[5],linewidth=0.5,color='b')
# ax.plot(Bef2[2],Bef2[3],linewidth=0.5,color='r')

# ax.plot(Bef3[2],Bef3[5],linewidth=0.5,color='b')
# ax.plot(Bef3[2],Bef3[3],linewidth=0.5,color='r')

# ax.plot(Bef4[2],Bef4[5],linewidth=0.5,color='b')
# ax.plot(Bef4[2],Bef4[3],linewidth=0.5,color='r')

# ax.plot(Bef5[2],Bef5[5],linewidth=0.5,color='b')
# ax.plot(Bef5[2],Bef5[3],linewidth=0.5,color='r')

# ax.plot(Bef5[2],Bef5[5],linewidth=0.5,color='b')
# ax.plot(Bef5[2],Bef5[3],linewidth=0.5,color='r')





ax.plot(Mov_ZR,T_R[0:i],linewidth=0.5,color='b') # Datos de los ejes y color
#ax.plot(Mov_Epgraf,T_Egraf,linewidth=0.5,color='r') # Datos de los ejes y color
#ax.plot(Mov_Epgraf,T_E[0:i],linewidth=0.5,color='k') # Datos de los ejes y color
ax.plot(Mov_ZE,T_E[0:i],linewidth=0.5,color='r') # Datos de los ejes y color
# ax.plot(Mov_Zs[-2],T_Rs[-2],linewidth=0.5,color='r') # Datos de los ejes y c
# ax.plot(Mov_Zs[-2],T_Es[-2],linewidth=0.5,color='r') # Datos de los ejes y color






ax.scatter(x_exp,y_exp, color='k',s=5)

ax.set(xlabel="Axial distance from nozzle [m]",
        ylabel="Refined and Extract phase Temperature [°C]")
        #title='Recorrido de la partícula')

ax.grid()
ax.grid(color='#CCCCCC', linestyle='dotted', linewidth=0.5)

#plt.axes().set_aspect('equal')  # Hace que las celdas tengan las mismas proporciones

for axis in ['top','bottom','left','right']:
    ax.spines[axis].set_linewidth(0.5) # This line is now indented


#ax.set_xlim([0, (mt.ceil(Mov_R[-1]))/2 ])
#ax.set_ylim([0, mt.ceil(Mov_Z[-1])])
#plt.gca().invert_yaxis()                       # Invierte el eje y

# Límites de los ejes:

#plt.yticks(np.arange(0, mt.ceil(Mov_Z[-1]), 0.25))
#plt.xticks(np.arange(0, mt.ceil(Mov_R[-1])/2, 0.05))
#plt.axis('scaled')

fig.savefig("Figure1.pdf", dpi=600)
plt.show()




# # Falta:
# #     Añadir la sección de gas nuevo en cada tanda.
# #       - Cálcular la distancia recorrida por el gas nuevo.       x
# #       - Hallar la distancia a la que llegaría dicho gas.        x
# #       - Expresarlo al inicio de los vectores de solución.       x
# #       - Verificar que todo el vector de distancias se traslade. x
# #       - Verificar solución para otros casos, es decir, al variar la cantidad de gas...
# #     Añadir las ecuaciones de energía.                           x
# #     Cambiar la masa de una gota en las ecuaciones de fuerza.    x
# #     Hacer las gráficas correctamente.

#%%

x_expH = [0,0.6,1.0,1.40]
y_expH = [0.010, 0.028, 0.034, 0.035]

Y_graf = Vector[5]

fig, ax = plt.subplots(1, figsize=(10,5))

#ax.plot(Mov_ZR,T_R[0:i],linewidth=0.5,color='b') # Datos de los ejes y color

ax.plot(Mov_ZE,Y_graf,linewidth=0.5,color='r') # Datos de los ejes y color

ax.scatter(x_expH,y_expH, color='k',s=5)

#ax.scatter(x_exp,y_exp, color='k',s=5)

ax.set(xlabel="Axial distance from nozzle [m]",
        ylabel="Extract humidity [kg/kg]")
        #title='Recorrido de la partícula')

ax.grid()
ax.grid(color='#CCCCCC', linestyle='dotted', linewidth=0.5)

#plt.axes().set_aspect('equal')  # Hace que las celdas tengan las mismas proporciones

for axis in ['top','bottom','left','right']:
    ax.spines[axis].set_linewidth(0.5) # This line is now indented


#ax.set_xlim([0, (mt.ceil(Mov_R[-1]))/2 ])
#ax.set_ylim([0, mt.ceil(Mov_Z[-1])])
#plt.gca().invert_yaxis()                       # Invierte el eje y

# Límites de los ejes:

#plt.yticks(np.arange(0, mt.ceil(Mov_Z[-1]), 0.25))
#plt.xticks(np.arange(0, mt.ceil(Mov_R[-1])/2, 0.05))
#plt.axis('scaled')

fig.savefig("Figure3.pdf", dpi=600)
plt.show()

#%%

X_graf = Vector[11]

fig, ax = plt.subplots(1, figsize=(10,5))

#ax.plot(Mov_ZR,T_R[0:i],linewidth=0.5,color='b') # Datos de los ejes y color

ax.plot(Mov_ZE,X_graf,linewidth=0.5,color='r') # Datos de los ejes y color


#ax.scatter(x_exp,y_exp, color='k',s=5)

ax.set(xlabel="Axial distance from nozzle [m]",
        ylabel="Refined moisture content [kg of water/kg of maltodextrin]")
        #title='Recorrido de la partícula')

ax.grid()
ax.grid(color='#CCCCCC', linestyle='dotted', linewidth=0.5)

#plt.axes().set_aspect('equal')  # Hace que las celdas tengan las mismas proporciones

for axis in ['top','bottom','left','right']:
    ax.spines[axis].set_linewidth(0.5) # This line is now indented


#ax.set_xlim([0, (mt.ceil(Mov_R[-1]))/2 ])
#ax.set_ylim([0, mt.ceil(Mov_Z[-1])])
#plt.gca().invert_yaxis()                       # Invierte el eje y

# Límites de los ejes:

#plt.yticks(np.arange(0, mt.ceil(Mov_Z[-1]), 0.25))
#plt.xticks(np.arange(0, mt.ceil(Mov_R[-1])/2, 0.05))
#plt.axis('scaled')

fig.savefig("Figure4.pdf", dpi=600)
plt.show()

#%%

D_graf = Vector[1]

fig, ax = plt.subplots(1, figsize=(10,5))

#ax.plot(Mov_ZR,T_R[0:i],linewidth=0.5,color='b') # Datos de los ejes y color

ax.plot(Mov_ZE,D_graf,linewidth=0.5,color='r') # Datos de los ejes y color


#ax.scatter(x_exp,y_exp, color='k',s=5)

ax.set(xlabel="Axial distance from nozzle [m]",
        ylabel="Droplet diameter [m]")
        #title='Recorrido de la partícula')

ax.grid()
ax.grid(color='#CCCCCC', linestyle='dotted', linewidth=0.5)

#plt.axes().set_aspect('equal')  # Hace que las celdas tengan las mismas proporciones

for axis in ['top','bottom','left','right']:
    ax.spines[axis].set_linewidth(0.5) # This line is now indented


#ax.set_xlim([0, (mt.ceil(Mov_R[-1]))/2 ])
#ax.set_ylim([0, mt.ceil(Mov_Z[-1])])
#plt.gca().invert_yaxis()                       # Invierte el eje y

# Límites de los ejes:

#plt.yticks(np.arange(0, mt.ceil(Mov_Z[-1]), 0.25))
#plt.xticks(np.arange(0, mt.ceil(Mov_R[-1])/2, 0.05))
#plt.axis('scaled')

fig.savefig("Figure5.pdf", dpi=600)
plt.show()

#%%




T_Eout = Vector[23]

for u in range(0,100):
    T_Eout[u] = 75

EjX = np.linspace(0,int(T*h_eu-1),int(T*h_eu))

fig, ax = plt.subplots(1, figsize=(10,5))

ax.plot(EjX,T_Eout,linewidth=0.5,color='b') # Datos de los ejes y color

ax.set(xlabel="Time [s]",
        ylabel="Outlet temperature [°C]")
        #title='Recorrido de la partícula')

ax.grid()
ax.grid(color='#CCCCCC', linestyle='dotted', linewidth=0.5)

#plt.axes().set_aspect('equal')  # Hace que las celdas tengan las mismas proporciones

for axis in ['top','bottom','left','right']:
    ax.spines[axis].set_linewidth(0.5) # This line is now indented


ax.set_xlim([0, mt.ceil(EjX[-1]) ])
#ax.set_ylim([0, mt.ceil(Mov_Z[-1])])
#plt.gca().invert_yaxis()                       # Invierte el eje y

# Límites de los ejes:

plt.yticks(np.arange(70, 90, 1))
plt.xticks(np.arange(0, mt.ceil(EjX[-1]), 500))
#plt.axis('scaled')

#l1=plt.axhline(113,linewidth=0.5,color='black',ls='--')
#l1.set_label('T1')

l2=plt.axhline(85,linewidth=0.5,color='black',ls='--')
l2.set_label('T2')


X_Exp2 = [250.068,
301.188,
318.684,
326.172,
330.564,
338.88,
348.096,
355.116,
370.884,
378.336,
387.516,
401.088,
423.408,
434.784,
447.456,
461.46,
485.94,
511.752,
546.276,
564.204,
600.924,
631.524,
662.124,
698.412,
742.116,
770.088,
785.388,
797.628,
826.5,
868.872,
901.236,
929.208,
965.064,
1000.452,
1025.796,
1075.62,
1111.44,
1148.196,
1199.712,
1233.408,
1272.72,
1315.128,
1379.352,
1444.512,
1479,
1530.588,
1566.408,
1610.544,
1655.148,
1692.3,
1725.06]

X_Exp2 = [400.136,
451.256,
468.752,
476.24,
480.632,
488.948,
498.164,
505.184,
520.952,
528.404,
537.584,
551.156,
573.476,
584.852,
597.524,
611.528,
636.008,
661.82,
696.344,
714.272,
750.992,
781.592,
812.192,
848.48,
892.184,
920.156,
935.456,
947.696,
976.568,
1018.94,
1051.304,
1079.276,
1115.132,
1150.52,
1175.864,
1225.688,
1261.508,
1298.264,
1349.78,
1383.476,
1422.788,
1465.196,
1529.42,
1594.58,
1629.068,
1680.656,
1716.476,
1760.612,
1805.216,
1842.368,
1875.128,
1917.968,
1954.256,
1987.448,
2045.12,
2078.78,
2115.068,
2165.324,
2205.536,
2250.968,
2286.392,
2357.168,
2392.556,
2456.384,
2510.564,
2541.164,
2610.644,
2657.84,
2699.384,
2753.132,
2801.192,
2868.944,
2908.256]

Y_Exp2 = [81.5272,
81.5487,
81.4156,
81.1238,
80.8307,
80.6007,
80.3453,
80.0656,
79.8276,
79.439,
79.2574,
79.0665,
78.9145,
78.7774,
78.647,
78.499,
78.3242,
78.2462,
78.14,
77.9867,
77.8912,
77.8347,
77.711,
77.6814,
77.6437,
77.5939,
77.5106,
77.4057,
77.325,
77.2497,
77.2053,
77.1219,
77.0681,
77.0546,
76.9484,
76.9282,
76.9954,
76.8071,
76.9967,
76.8124,
76.8501,
76.721,
76.7881,
76.5609,
76.6644,
76.5743,
76.8203,
76.6307,
76.5123,
76.4854,
76.671]

Y_Exp2 = [81.5272,
81.5487,
81.4156,
81.1238,
80.8307,
80.6007,
80.3453,
80.0656,
79.8276,
79.439,
79.2574,
79.0665,
78.9145,
78.7774,
78.647,
78.499,
78.3242,
78.2462,
78.14,
77.9867,
77.8912,
77.8347,
77.711,
77.6814,
77.6437,
77.5939,
77.5106,
77.4057,
77.325,
77.2497,
77.2053,
77.1219,
77.0681,
77.0546,
76.9484,
76.9282,
76.9954,
76.8071,
76.9967,
76.8124,
76.8501,
76.721,
76.7881,
76.5609,
76.6644,
76.5743,
76.8203,
76.6307,
76.5123,
76.4854,
76.671,
76.5015,
76.3536,
76.3146,
76.4261,
76.3293,
76.3925,
76.3817,
76.301,
76.4408,
76.2041,
76.3211,
76.4138,
76.2807,
76.2363,
76.1784,
76.4016,
76.2133,
76.0842,
76.1608,
76.0479,
76.1124,
76.2602]



X_Exp3 = [1949.456,
1991.864,
2018.504,
2030.276,
2035.928,
2047.268,
2051.156,
2053.316,
2060.696,
2065.916,
2075.06,
2078.516,
2095.076,
2101.628,
2103.32,
2118.152,
2123.372,
2140.832,
2150.876,
2167.004,
2180.54,
2195.408,
2199.728,
2216.756,
2238.176,
2266.112,
2291.888,
2303.696,
2317.664,
2325.08,
2360.468,
2367.848,
2383.58,
2413.712,
2429.876,
2439.488,
2446.904,
2464.796,
2492.336,
2522.036,
2539.532]

X_Exp3 = [2449.456,
2491.864,
2518.504,
2530.276,
2535.928,
2547.268,
2551.156,
2553.316,
2560.696,
2565.916,
2575.06,
2578.516,
2595.076,
2601.628,
2603.32,
2618.152,
2623.372,
2640.832,
2650.876,
2667.004,
2680.54,
2695.408,
2699.728,
2716.756,
2738.176,
2766.112,
2791.888,
2803.696,
2817.664,
2825.08,
2860.468,
2867.848,
2883.58,
2913.712,
2929.876,
2939.488,
2946.904,
2964.796,
2992.336,
3022.036,
3039.532,
3064.876,
3076.648,
3098.5,
3118.156,
3151.78,
3171.004,
3193.72,
3213.376,
3235.228,
3267.556,
3285.016,
3321.736,
3367.6,
3407.812,
3442.3,
3462.856,
3488.2,
3544.144,
3578.236,
3600.052,
3635.44,
3684.4,
3739.012,
3764.752,
3809.356,
3855.22,
3905.044,
3969.268,
4006.852,
4068.916,
4130.512,
4179.904,
4235.416,
4291.756,
4345.504,
4390.936,
4445.584,
4494.94]

Y_Exp3 = [76.0135,
75.9328,
76.1009,
76.3496,
76.7624,
77.0824,
77.3177,
77.5584,
77.9457,
78.2105,
78.7618,
79.1639,
79.3857,
79.6506,
79.9827,
80.21,
80.5246,
80.6725,
80.8164,
81.0301,
81.1767,
81.3596,
81.5854,
81.7696,
81.9149,
81.9834,
82.0842,
82.2214,
82.3895,
82.5132,
82.6153,
82.77,
82.8883,
83.0079,
83.1383,
83.2634,
83.4368,
83.5444,
83.5847,
83.5215,
83.4233]

Y_Exp3 = [76.0135,
75.9328,
76.1009,
76.3496,
76.7624,
77.0824,
77.3177,
77.5584,
77.9457,
78.2105,
78.7618,
79.1639,
79.3857,
79.6506,
79.9827,
80.21,
80.5246,
80.6725,
80.8164,
81.0301,
81.1767,
81.3596,
81.5854,
81.7696,
81.9149,
81.9834,
82.0842,
82.2214,
82.3895,
82.5132,
82.6153,
82.77,
82.8883,
83.0079,
83.1383,
83.2634,
83.4368,
83.5444,
83.5847,
83.5215,
83.4233,
83.6035,
83.7796,
83.9087,
84.0216,
84.1157,
84.2381,
84.3295,
84.4169,
84.4935,
84.4491,
84.6212,
84.6737,
84.7395,
84.7139,
84.8524,
84.8954,
84.999,
84.8793,
84.8416,
85.0769,
85.2745,
85.1615,
85.3242,
85.4519,
85.3484,
85.4304,
85.5715,
85.4599,
85.6037,
85.6736,
85.6494,
85.7408,
85.6789,
85.7192,
85.7985,
85.906,
86.019,
85.9638]


ax.scatter(X_Exp2,Y_Exp2, color='k',s=5)
ax.scatter(X_Exp3,Y_Exp3, color='k',s=5)

plt.axvline(x=80,linewidth=0.5,color='red',linestyle='-')

#plt.axvline(x=200,linewidth=0.5,color='red',linestyle='-')

fig.savefig("Figure6.pdf", dpi=600)
plt.show()


#%%

Y_Eout = Vector[24]
EjX = np.linspace(0,int(T*h_eu-1),int(T*h_eu))

fig, ax = plt.subplots(1, figsize=(10,5))

ax.plot(EjX,Y_Eout,linewidth=0.5,color='b') # Datos de los ejes y color

ax.set(xlabel="Time [s]",
        ylabel="Humidity of outlet air [kg/kg]")
        #title='Recorrido de la partícula')

ax.grid()
ax.grid(color='#CCCCCC', linestyle='dotted', linewidth=0.5)

#plt.axes().set_aspect('equal')  # Hace que las celdas tengan las mismas proporciones

for axis in ['top','bottom','left','right']:
    ax.spines[axis].set_linewidth(0.5) # This line is now indented


ax.set_xlim([0, mt.ceil(EjX[-1]) ])
#ax.set_ylim([0, mt.ceil(Mov_Z[-1])])
#plt.gca().invert_yaxis()                       # Invierte el eje y

# Límites de los ejes:

#plt.yticks(np.arange(0, mt.ceil(Mov_Z[-1]), 0.25))
plt.xticks(np.arange(0, mt.ceil(EjX[-1]), 25))
#plt.axis('scaled')

# l1=plt.axhline(113,linewidth=0.5,color='black',ls='--')
# l1.set_label('T1')

# l2=plt.axhline(85,linewidth=0.5,color='black',ls='--')
# l2.set_label('T2')

plt.axvline(x=80,linewidth=0.5,color='red',linestyle='-')
#plt.axvline(x=200,linewidth=0.5,color='red',linestyle='-')

fig.savefig("Figure7.pdf", dpi=600)
plt.show()


#%%

Time_exp = [143.800000000000,199.060000000000,245.968000000000,267.244000000000,300.004000000000,313.792000000000,318.364000000000,330.964000000000,337.336000000000,355.732000000000,363.868000000000	,389.284000000000,426.544000000000,476.872000000000,586.996000000000,643.408000000000,689.416000000000]

T_exp = [170.6506,
170.3273,
169.8626,
169.4455,
169.128,
168.4857,
167.6659,
166.1259,
165.0751,
163.4095,
162.2563,
160.8952,
160.0451,
160.2126,
159.5228,
160.3326,
160.0007]

Y_Eout = Vector[25]
EjX = np.linspace(0,int(T*h_eu-1),int(T*h_eu))

fig, ax = plt.subplots(1, figsize=(10,5))

ax.plot(EjX,Y_Eout,linewidth=0.5,color='b') # Datos de los ejes y color

ax.scatter(Time_exp,T_exp, color='k',s=5)

ax.set(xlabel="Time [s]",
        ylabel="Inlet temperature [°C]")
        #title='Recorrido de la partícula')

ax.grid()
ax.grid(color='#CCCCCC', linestyle='dotted', linewidth=0.5)

#plt.axes().set_aspect('equal')  # Hace que las celdas tengan las mismas proporciones

for axis in ['top','bottom','left','right']:
    ax.spines[axis].set_linewidth(0.5) # This line is now indented


ax.set_xlim([0, mt.ceil(EjX[-1]) ])
#ax.set_ylim([0, mt.ceil(Mov_Z[-1])])
#plt.gca().invert_yaxis()                       # Invierte el eje y

# Límites de los ejes:

#plt.yticks(np.arange(0, mt.ceil(Mov_Z[-1]), 0.25))
plt.xticks(np.arange(0, mt.ceil(EjX[-1]), 100))
#plt.axis('scaled')

# l1=plt.axhline(113,linewidth=0.5,color='black',ls='--')
# l1.set_label('T1')

# l2=plt.axhline(85,linewidth=0.5,color='black',ls='--')
# l2.set_label('T2')

plt.axvline(x=80,linewidth=0.5,color='red',linestyle='-')
#plt.axvline(x=200,linewidth=0.5,color='red',linestyle='-')

fig.savefig("Figure8.pdf", dpi=600)
plt.show()

#%%

Z_Rout = Vector[26]
EjX = np.linspace(0,int(T*h_eu-1),int(T*h_eu))

fig, ax = plt.subplots(1, figsize=(10,5))

ax.plot(EjX,Z_Rout,linewidth=0.5,color='b') # Datos de los ejes y color

ax.set(xlabel="Time [s]",
        ylabel="Moisture content [kg/kg]")
        #title='Recorrido de la partícula')

ax.grid()
ax.grid(color='#CCCCCC', linestyle='dotted', linewidth=0.5)

#plt.axes().set_aspect('equal')  # Hace que las celdas tengan las mismas proporciones

for axis in ['top','bottom','left','right']:
    ax.spines[axis].set_linewidth(0.5) # This line is now indented


ax.set_xlim([0, mt.ceil(EjX[-1]) ])
#ax.set_ylim([0, mt.ceil(Mov_Z[-1])])
#plt.gca().invert_yaxis()                       # Invierte el eje y

# Límites de los ejes:

#plt.yticks(np.arange(0, mt.ceil(Mov_Z[-1]), 0.25))
plt.xticks(np.arange(0, mt.ceil(EjX[-1]), 25))
#plt.axis('scaled')

# l1=plt.axhline(113,linewidth=0.5,color='black',ls='--')
# l1.set_label('T1')

# l2=plt.axhline(85,linewidth=0.5,color='black',ls='--')
# l2.set_label('T2')

plt.axvline(x=80,linewidth=0.5,color='red',linestyle='-')
#plt.axvline(x=200,linewidth=0.5,color='red',linestyle='-')

fig.savefig("Figure9.pdf", dpi=600)
plt.show()

#%%

P_inMV = Vector[27]
EjX = np.linspace(0,int(T*h_eu-1),int(T*h_eu))

fig, ax = plt.subplots(1, figsize=(10,5))

ax.plot(EjX,P_inMV,linewidth=0.5,color='b') # Datos de los ejes y color

ax.set(xlabel="Time [s]",
        ylabel="Inlet pressure of the liquid [Pa]")
        #title='Recorrido de la partícula')

ax.grid()
ax.grid(color='#CCCCCC', linestyle='dotted', linewidth=0.5)

#plt.axes().set_aspect('equal')  # Hace que las celdas tengan las mismas proporciones

for axis in ['top','bottom','left','right']:
    ax.spines[axis].set_linewidth(0.5) # This line is now indented


ax.set_xlim([0, mt.ceil(EjX[-1]) ])
#ax.set_ylim([0, mt.ceil(Mov_Z[-1])])
#plt.gca().invert_yaxis()                       # Invierte el eje y

# Límites de los ejes:

#plt.yticks(np.arange(0, mt.ceil(Mov_Z[-1]), 0.25))
plt.xticks(np.arange(0, mt.ceil(EjX[-1]), 25))
#plt.axis('scaled')

# l1=plt.axhline(113,linewidth=0.5,color='black',ls='--')
# l1.set_label('T1')

# l2=plt.axhline(85,linewidth=0.5,color='black',ls='--')
# l2.set_label('T2')

plt.axvline(x=80,linewidth=0.5,color='red',linestyle='-')
#plt.axvline(x=200,linewidth=0.5,color='red',linestyle='-')

fig.savefig("Figure10.pdf", dpi=600)
plt.show()

#%%

m_LMV = Vector[28]
EjX = np.linspace(0,int(T*h_eu-1),int(T*h_eu))

fig, ax = plt.subplots(1, figsize=(10,5))

ax.plot(EjX,m_LMV,linewidth=0.5,color='b') # Datos de los ejes y color

ax.set(xlabel="Time [s]",
        ylabel="Liquid flow [kg/h]")
        #title='Recorrido de la partícula')

ax.grid()
ax.grid(color='#CCCCCC', linestyle='dotted', linewidth=0.5)

#plt.axes().set_aspect('equal')  # Hace que las celdas tengan las mismas proporciones

for axis in ['top','bottom','left','right']:
    ax.spines[axis].set_linewidth(0.5) # This line is now indented


ax.set_xlim([0, mt.ceil(EjX[-1]) ])
#ax.set_ylim([0, mt.ceil(Mov_Z[-1])])
#plt.gca().invert_yaxis()                       # Invierte el eje y

# Límites de los ejes:

#plt.yticks(np.arange(0, mt.ceil(Mov_Z[-1]), 0.25))
plt.xticks(np.arange(0, mt.ceil(EjX[-1]), 25))
#plt.axis('scaled')

# l1=plt.axhline(113,linewidth=0.5,color='black',ls='--')
# l1.set_label('T1')

# l2=plt.axhline(85,linewidth=0.5,color='black',ls='--')
# l2.set_label('T2')

plt.axvline(x=80,linewidth=0.5,color='red',linestyle='-')
#plt.axvline(x=200,linewidth=0.5,color='red',linestyle='-')

fig.savefig("Figure11.pdf", dpi=600)
plt.show()







#%% Cálculo de verificación:

#m_ss = 0.005912783767187386
#m_da = 0.3144037221752935

m_ss = (82.2/3600)*(1-0.4)

T_E0 = 170
rho_air0 = 1.293*(273.15)/(273.15+T_E0) # [kg/m^3]

m_air = 1750  # kg/h
Vol_air = m_air/rho_air0                 # [m^3/s] ... (1750(kg/h)/rho_air0(kg/m^3))/3600

X_graf = Vector[11]

m_da = 0.4846172128430196

Izq = Y_w[0]*m_da + m_ss*X_graf[0]
Der = Y_w[-1]*m_da + m_ss*X_graf[-1]

f1 = (0.003/1.003)*1750 + 82.2*(0.6) - (1-0.003)*1750*Y_w[-1] - (0.6*82.2)*X_graf[-1] # [kg/h]

f_transf1 = 82.2*(0.6) - (0.6*82.2)*X_graf[-1]
f_transf2 = (0.003/1.003)*1750 - (1-0.003)*1750*Y_w[-1]

f2 = (0.003/1.003)*1710 + 88.2*(0.5) - (1-0.003)*1710*Y_w[-1] - (0.5*88.2)*X_graf[-1] # [kg/h]

KeyboardInterrupt: 